In [1]:
# ==========================================================
# Install
# ==========================================================

!pip -q install optuna

In [2]:
# ==========================================================
# Imports
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import optuna

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from xgboost import XGBClassifier

In [3]:

DATA = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(DATA)

In [4]:
TARGET = "is_correct"

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].astype("category").cat.codes

print(X.shape)

(35072, 113)


In [5]:
# ==========================================================
# Optuna Objective
# ==========================================================

def objective(trial):

    params = {

        "objective":"binary:logistic",

        "eval_metric":"logloss",

        "tree_method":"hist",

        "random_state":42,

        "n_estimators":4000,

        "learning_rate":trial.suggest_float(
            "learning_rate",
            0.01,
            0.03
        ),

        "max_depth":trial.suggest_int(
            "max_depth",
            6,
            8
        ),

        "min_child_weight":trial.suggest_int(
            "min_child_weight",
            1,
            5
        ),

        "subsample":trial.suggest_float(
            "subsample",
            0.7,
            0.9
        ),

        "colsample_bytree":trial.suggest_float(
            "colsample_bytree",
            0.5,
            0.8
        ),

        "gamma":trial.suggest_float(
            "gamma",
            0,
            3
        ),

        "reg_alpha":trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda":trial.suggest_float(
            "reg_lambda",
            0.5,
            10
        ),

        "early_stopping_rounds":300,
    }

    kf = StratifiedKFold(

        n_splits=5,

        shuffle=True,

        random_state=42,

    )

    oof = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X,y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(**params)

        model.fit(

            X_train,
            y_train,

            eval_set=[(X_valid,y_valid)],

            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof[valid_idx] = pred

    score = log_loss(y,oof)

    return score

In [6]:
study = optuna.create_study(

    direction="minimize",

    study_name="XGB_LogLoss",
)

study.optimize(

    objective,

    n_trials=50,

    show_progress_bar=True,
)

[I 2026-08-04 02:10:43,415] A new study created in memory with name: XGB_LogLoss


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-04 02:11:51,083] Trial 0 finished with value: 0.5444617108080322 and parameters: {'learning_rate': 0.02387533938260733, 'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.7908197209523926, 'colsample_bytree': 0.6095814893884705, 'gamma': 0.585453794576277, 'reg_alpha': 4.577688176411941, 'reg_lambda': 4.080612993630915}. Best is trial 0 with value: 0.5444617108080322.
[I 2026-08-04 02:12:47,659] Trial 1 finished with value: 0.5440968690294212 and parameters: {'learning_rate': 0.025276487713604356, 'max_depth': 6, 'min_child_weight': 4, 'subsample': 0.8783687237482236, 'colsample_bytree': 0.5864949150695697, 'gamma': 0.709177650926907, 'reg_alpha': 4.231587553435818, 'reg_lambda': 9.731736702010556}. Best is trial 1 with value: 0.5440968690294212.
[I 2026-08-04 02:14:07,678] Trial 2 finished with value: 0.5450757489493054 and parameters: {'learning_rate': 0.01607202102027481, 'max_depth': 6, 'min_child_weight': 4, 'subsample': 0.7290175350551771, 'colsample_bytree': 0.6819

In [7]:
print("="*60)

print("Best LogLoss")

print(study.best_value)

print()

print("Best Parameters")

print(study.best_params)

print("="*60)

Best LogLoss
0.5424860717247075

Best Parameters
{'learning_rate': 0.011657630809581768, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.897872380787519, 'colsample_bytree': 0.6307436716260597, 'gamma': 2.2072247646895793, 'reg_alpha': 2.1589558695750513, 'reg_lambda': 1.933804805181796}


**learning_rate      = 0.01379  
max_depth          = 8  
min_child_weight   = 2  
 
subsample          = 0.8821  
colsample_bytree   = 0.7130  

gamma              = 1.6320

reg_alpha          = 0.4615  
reg_lambda         = 2.5213**